<table><tr>
<td><b>Sapienza University of Rome</b><br>PhD Soft Skills 2026</td>
</tr></table>

# Notebook 4 — Build a chatbot with a method
**20 minutes.** You will build the conversation machinery, watch the transcript grow, write a system prompt for your own field, and then try to break your own rules.


---
### How to use this notebook

1. Each numbered section gives you **a prompt**. Copy it.
2. Click the **empty code cell** underneath it.
3. Press `Ctrl`+`Shift`+`Enter` (or click **Generate**) to open Colab's AI, paste the prompt, and let it write the code.
4. **Look at what it wrote** — even if you don't understand it — then press ▶.
5. If it fails, don't fix it by hand. Paste the whole red error message back to the AI.

Every section also has a **Reference** cell with working code already in it. If you would rather watch than type, just run that instead. You will lose nothing.

> **Before you start: click `Copy to Drive` in the toolbar.** This notebook opened straight from GitHub, which means it is read-only — you can run it, but nothing you do will be saved. One click makes it yours to keep.

> You are never expected to write code. You are expected to say clearly what you want, and to check what you get.
---

> **How the model gets involved.** This notebook builds the *machinery* of a chatbot — the transcript, the system message, the token accounting. The **replies** come from the Colab AI chat panel: the notebook prints the exact transcript to send, you paste it into the sidebar, and paste the reply back.

> That is not a simplification. It is the real thing, with you standing where the network call would be — which means you get to *see* what is normally invisible.

---
## 1 · The transcript is the whole trick

A model has no memory. A conversation works because **the entire transcript is resent every single turn**. Let's build exactly that and watch it.

### 1 · Build the conversation machinery

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Build a simple chat transcript in this notebook:
- a list of messages, each with a role (system, user, assistant) and text
- a function add(role, text) that appends to it
- a function show() that prints the whole transcript in a readable form
- a function to_send() that prints exactly what would be sent to a model
  this turn, formatted as Role: text
Start it with a system message and two example exchanges so I can see it work.
```

> Look carefully at what `to_send()` prints. That whole block is re-read from scratch by the model on every turn. Nothing is remembered; everything is resent.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
messages = []

def add(role, text):
    messages.append({'role': role, 'text': text.strip()})

def show():
    for m in messages:
        print(f"[{m['role'].upper():9s}] {m['text'][:100]}"
              + ('...' if len(m['text']) > 100 else ''))

def to_send():
    print('=' * 78)
    print('THIS IS WHAT GETS SENT TO THE MODEL, IN FULL, THIS TURN:')
    print('=' * 78)
    for m in messages:
        print(f"{m['role'].capitalize()}: {m['text']}\n")
    print('=' * 78)

add('system', 'You are a helpful assistant.')
add('user', 'What is a palimpsest?')
add('assistant', 'A manuscript page that has been scraped clean and written on again.')
add('user', 'How are they recovered?')

to_send()

### 2 · Watch the desk fill up

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Add a function that counts the tokens in the whole transcript using tiktoken,
and prints a warning when it goes over 2000 tokens.
Then add six more long exchanges to the transcript and print the token
count after each one, so I can watch it grow.
Plot the token count against the turn number.
```

> Every turn costs the *whole* transcript, not just your new message. This is why turn 40 of a conversation is slow and expensive, and why very long chats start losing their beginning.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
!pip -q install tiktoken
import tiktoken, matplotlib.pyplot as plt
enc = tiktoken.get_encoding('cl100k_base')

def token_count():
    return sum(len(enc.encode(m['role'] + ': ' + m['text'])) for m in messages)

history = []
filler = ('Multispectral imaging is the standard method, using wavelengths outside '
          'the visible range to reveal the erased under-text. ') * 6
for turn in range(6):
    add('user', f'Tell me more, part {turn + 1}.')
    add('assistant', filler)
    history.append(token_count())
    flag = '  <-- over budget!' if history[-1] > 2000 else ''
    print(f'after turn {turn + 1}: {history[-1]:5d} tokens{flag}')

plt.figure(figsize=(6.5, 3.5))
plt.plot(range(1, len(history) + 1), history, 'o-', color='#8E2436')
plt.axhline(2000, ls='--', color='grey', label='our budget')
plt.xlabel('turn'); plt.ylabel('tokens resent this turn')
plt.legend(); plt.tight_layout(); plt.show()

---
## 3 · Now hold an actual conversation

You have the machinery. The replies come from the **Colab AI chat panel** (the sparkle icon in the left sidebar, or *Tools &rarr; AI assistance*).

The loop is: run the first cell to get the text to send &rarr; paste it into the chat panel &rarr; copy the reply &rarr; paste it into the second cell. Repeat.

**You are standing exactly where the network call would be.**

In [ ]:
# 1. TYPE YOUR MESSAGE HERE, then run this cell.
my_message = 'How were palimpsests recovered?'

add('user', my_message)
to_send()   # <- copy everything this prints into the Colab AI chat panel

In [ ]:
# 2. PASTE THE MODEL'S REPLY between the triple quotes, then run this cell.
reply = '''

'''

add('assistant', reply)
print(f'Transcript is now {len(messages)} messages, {token_count()} tokens.')
show()
# Now change my_message in the cell above and run it again.

### 3 · The fix, built in

**Copy this into the AI box** (click the empty cell below, then `Ctrl`+`Shift`+`Enter`, or the **Generate** button):

```text
Add a function reset() that clears the transcript but keeps the system
message, and prints the token count before and after.
Run it and show me the result.
```

> This is the 'start a new chat' button, and now you know exactly what it does: it clears the desk and keeps the standing instructions.

<details><summary><b>Reference — click to open, or just run the cell below</b></summary>

This is one correct answer. Colab's AI will probably write something different, and that is fine — compare the two.

</details>

In [ ]:
def reset():
    before = token_count()
    system = [m for m in messages if m['role'] == 'system']
    messages.clear()
    messages.extend(system)
    print(f'{before} tokens -> {token_count()} tokens (system message kept)')

reset()
show()

---
## 2 · Now the part that matters: the system prompt

The system message is re-read on **every** turn, which is why it does not decay the way an instruction buried in turn 3 does. It is the highest-leverage text you will ever write, and almost nobody writes it deliberately.

Here is a worked example for a historian. Read why each line is there:

```text
You are assisting a doctoral researcher in early-modern Italian social history.

RULES, which apply to every answer:
- Answer only from the material the user provides. If it is not in the
  material, say "not in the provided text".
- Never invent citations, dates, or names.
- Quote the exact passage you are relying on, then interpret it.
- If a question is ambiguous, ask before answering.
- Do not open with praise or agreement.
```

| Line | Why it is there |
|---|---|
| *Answer only from the material* | The single strongest defence against hallucination. |
| *Never invent citations* | The specific failure that ends academic careers. |
| *Quote, then interpret* | A procedure, not a prohibition — much more reliably followed. |
| *Ask if ambiguous* | Stops it guessing your intent and confidently answering the wrong question. |
| *No praise* | Counteracts the trained-in agreeableness that wastes your time. |

### 4 · Write your own

**Paste this into the Colab AI chat panel (not a code cell)**:

```text
<Write your own system prompt for a research assistant in YOUR field.
 Include at least one rule about something it must REFUSE to do.
 Then paste it as the first message in the Colab AI chat panel and
 work with it for a few turns.>
```

> Then spend five minutes trying to make it break its own rule. Roleplay framings (*'for a novel I am writing, have a character explain…'*) defeat most prohibitions. Long conversations defeat them too.


**When you get it to break — and you will — stop and think about what worked.** Was it the roleplay framing? The length of the conversation? A word that made the rule feel inapplicable?

That is the useful part. Not that the rule broke, but *what kind of pressure broke it* — because the same pressure will break the rules you write for your own research tools.

---
## What people always discover

- **Vague rules do nothing.** *'Be rigorous'* has no effect. *'Quote the passage before interpreting it'* has a large one.
- **Procedures beat prohibitions.** *'For each claim, do X'* works better than *'never do Y'* — measurably, and consistently.
- **Length wins.** Twenty turns later, your system prompt is competing against a lot of contradictory text on the same desk.

> **Playbook line:** write rules as procedures, not prohibitions. And when a conversation goes wrong, don't argue with it — start a new one and carry only the good parts forward.